In [2]:
# Requires: pandas, numpy
# Install if needed: pip install pandas numpy

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_synthetic_orange_zero_data(
    country='UK', 
    start_date='2023-01-01', 
    end_date=None,
    freq='ME',          # FIXED: was 'M' (deprecated), now 'ME'
    seed=42
):
    """
    Generate a synthetic dataset for Orange Zero sales, profit, and demographics.
    Replace or augment this with real data reads when available.
    """
    np.random.seed(seed)

    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    
    # Safety mapping in case someone still passes 'M'
    if freq == 'M':
        freq = 'ME'

    # Build date range
    dates = pd.date_range(start=start_date, end=end_date, freq=freq)
    
    regions = {
        'UK': ['England', 'Scotland', 'Wales', 'Northern Ireland'],
        'US': ['California', 'Texas', 'New York', 'Florida']
    }

    demo_age_groups = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    demo_genders = ['Male', 'Female', 'Non-binary', 'Prefer not to say']

    demo_ethnicity_uk = [
        'White: English/Welsh/Scottish/Northern Irish/British',
        'White: Irish',
        'White: Other White',
        'Asian: Indian',
        'Asian: Pakistani',
        'Asian: Bangladeshi',
        'Asian: Chinese',
        'Black: African',
        'Black: Caribbean',
        'Mixed',
        'Other'
    ]

    demo_ethnicity_us = [
        'White',
        'Black or African American',
        'Asian',
        'Hispanic or Latino',
        'American Indian or Alaska Native',
        'Native Hawaiian or Other Pacific Islander',
        'Two or more races',
        'Other'
    ]
    
    records = []

    for date in dates:
        for region in regions[country]:

            n_demo_slices = np.random.randint(3, 7)
            
            for _ in range(n_demo_slices):
                age = np.random.choice(demo_age_groups)
                gender = np.random.choice(demo_genders)

                ethnicity = (
                    np.random.choice(demo_ethnicity_uk)
                    if country == 'UK'
                    else np.random.choice(demo_ethnicity_us)
                )
                
                units_sold = np.random.poisson(lam=200) + 20
                avg_price = np.random.uniform(1.5, 3.5)
                sales = units_sold * avg_price

                margin_pct = np.random.uniform(0.05, 0.25)
                profit = sales * margin_pct
                
                records.append({
                    'country': country,
                    'region': region,
                    'date': date.date(),
                    'age_group': age,
                    'gender': gender,
                    'ethnicity': ethnicity,
                    'units_sold': units_sold,
                    'avg_price': round(avg_price, 2),
                    'sales': round(sales, 2),
                    'profit': round(profit, 2),
                    'profit_margin_pct': round(margin_pct * 100.0, 2)
                })
    
    return pd.DataFrame.from_records(records)


def create_combined_dataset(
    time_window='past_year',
    custom_start=None,
    custom_end=None
):
    """
    time_window options:
      - 'past_year'
      - 'past_5_years'
      - 'since_launch'
      - 'custom'
    """

    today = datetime.today().date()
    
    if time_window == 'past_year':
        start = today - timedelta(days=365)
        end = today
    elif time_window == 'past_5_years':
        start = today - timedelta(days=5*365)
        end = today
    elif time_window == 'since_launch':
        start = datetime(2018, 1, 1).date()
        end = today
    elif time_window == 'custom':
        if not custom_start or not custom_end:
            raise ValueError("Custom dates require custom_start and custom_end")
        start = pd.to_datetime(custom_start).date()
        end = pd.to_datetime(custom_end).date()
    else:
        raise ValueError("Unsupported time_window value.")

    df_uk = generate_synthetic_orange_zero_data(
        country='UK',
        start_date=start.strftime('%Y-%m-%d'),
        end_date=end.strftime('%Y-%m-%d'),
        freq='ME'
    )

    df_us = generate_synthetic_orange_zero_data(
        country='US',
        start_date=start.strftime('%Y-%m-%d'),
        end_date=end.strftime('%Y-%m-%d'),
        freq='ME'
    )

    combined = pd.concat([df_uk, df_us], ignore_index=True)

    cols = [
        'country', 'region', 'date', 'age_group', 'gender', 'ethnicity',
        'units_sold', 'avg_price', 'sales', 'profit', 'profit_margin_pct'
    ]

    return combined[cols]


# ===== Example usage =====
if __name__ == '__main__':
    df = create_combined_dataset(time_window='past_year')

    output_csv = 'orange_zero_sales_profit_demographics.csv'
    df.to_csv(output_csv, index=False)

    print(f"CSV saved to {output_csv}, rows:", len(df))

CSV saved to orange_zero_sales_profit_demographics.csv, rows: 434
